# Matrices bohemianas con población [0,1,2,...N], distribución singular y no singular

El estudio de las matrices Bohemianas (acrónimo de Bounded Height Matrix of Integers) ha emergido en la última década como un campo fértil en la intersección del álgebra lineal numérica, la combinatoria y la teoría de números. Si bien las matrices aleatorias con entradas continuas (Gaussianas, por ejemplo) han sido estudiadas exhaustivamente desde los trabajos de Wigner y Dyson, las matrices con entradas discretas y acotadas presentan comportamientos espectrales y estructurales únicos que desafían la intuición clásica. En el continuo, la probabilidad de encontrar una matriz singular es cero; en el dominio discreto Bohemiano, la singularidad es un evento de probabilidad finita y estructurada, cuya distribución encierra información profunda sobre la naturaleza aritmética de las transformaciones lineales.

El presente reporte tiene como objetivo diseccionar la metodología para el análisis de una familia específica de matrices Bohemianas: aquellas matrices cuadradas de dimensión $N \times N$ cuyas entradas pertenecen a la población $P_N = \{0, 1, 2, \dots, N\}$. Si se fuera a establecer $N=q-1$, siendo $q$ un número primo, se podría relacionar esto con el ámbito de la criptografía basada en retículos. A diferencia de las poblaciones $\{0, 1\}$ o ternarias $\{-1, 0, 1\}$ comúnmente estudiadas, la población $P_N$ escala con la dimensión de la matriz, introduciendo una complejidad factorial en el espacio de búsqueda $\Omega_N$, cuyo cardinal $|\Omega_N| = (N+1)^{N^2}$ crece a tasas superexponenciales. El avance presentado en este documento muestra la distribución de matrices no singulares y singulares, los tipos de singulares y logra generalizar sobre ciertos aspectos de combinatoria relacionados a las singularidades de carácter estructural. 

Este documento sirve como bitácora viva de la investigación sobre la distribución de determinantes y la densidad de singularidad en matrices con entradas discretas acotadas (Bohemianas).

**Objetivos:**

* Validación Numérica: Utilizar métodos de Monte Carlo y fuerza bruta paralelizada para obtener estadísticas empíricas.

* Formalización Analítica: Deducir fórmulas cerradas basadas en combinatoria (Principio de Inclusión-Exclusión y Números de Bell) para categorizar las estructuras de singularidad (filas nulas, columnas nulas, filas idénticas) sin necesidad de generarlas.

Cada sección viene con un código acompañado de una explicación asociada a la metodología y un análisis breve de los resultados obtenidos

## Aproximación Estocástica Inicial: Monte Carlo Paralelizado

Antes de abordar la complejidad analítica, establecemos una línea base numérica. Para dimensiones $N$ grandes, el espacio $\Omega_N$ es intratable ($N=5 \implies 6^{25}$ matrices). Por ello, implementamos un generador de Monte Carlo masivo.
### Lógica de la Implementación

El siguiente código implementa un esquema Map-Reduce utilizando joblib.
* Gestión de Memoria: No generamos listas gigantes. Procesamos por "lotes" (BATCH_SIZE) que caben cómodamente en la caché L3 del procesador.
* Vectorización: En lugar de bucles nativos de Python, generamos tensores $(B, N, N)$ y usamos las rutinas de LAPACK optimizadas de NumPy (np.linalg.det) para calcular $100,000$ determinantes simultáneamente.
* Estabilidad Numérica: Dado que trabajamos con enteros, cualquier determinante como 3.00000000004 es un error de punto flotante. Aplicamos redondeo y normalización de signo (-0.0 a 0.0) para asegurar la unicidad de las claves en el histograma.


In [2]:
import numpy as np
import pandas as pd
from collections import Counter
from joblib import Parallel, delayed
import time
import os

# --- CONFIGURACIÓN DE PARÁMETROS GLOBALES ---
# Definición de la magnitud de la simulación y límites de memoria
TOTAL_MUESTRAS = 1_000_000_000  # Objetivo: 1 Billón de matrices (10^9)
LOTE_SEGURO = 10_000_000        # Tamaño del buffer para evitar saturación de RAM

def simular_chunk(cantidad, n_dim):
    """
    Núcleo de la simulación: Genera un subconjunto de matrices y calcula sus determinantes.
    
    Args:
        cantidad (int): Número de matrices a generar en este hilo.
        n_dim (int): Dimensión N de la matriz cuadrada.
        
    Returns:
        dict: Diccionario comprimido {valor_determinante: frecuencia}.
    """
    # Generación vectorizada de matrices aleatorias con entradas {0,..., n_dim}
    matrices = np.random.randint(0, n_dim + 1, size=(cantidad, n_dim, n_dim), dtype=np.int8)
    
    # Cálculo numérico del determinante usando descomposición LU (backend de numpy)
    dets = np.linalg.det(matrices)
    
    # Corrección de errores de punto flotante
    dets = np.round(dets, 1)
    dets[dets == -0.0] = 0.0  # Normalización del cero negativo
    
    # Conteo de frecuencias (histograma parcial)
    unique, counts = np.unique(dets, return_counts=True)
    return dict(zip(unique, counts))

def ejecutar_monte_carlo_gigante(n, writer, col_inicio):
    """
    Orquestador del proceso paralelo. Gestiona la división del trabajo en lotes
    y consolida los resultados progresivos.
    """
    print(f"\n--- INICIANDO SIMULACIÓN PARA N={n} :: OBJETIVO: {TOTAL_MUESTRAS:,} MUESTRAS ---")
    
    total_counts = Counter()
    procesado_actual = 0
    proximo_hito = 10  # Porcentaje inicial para reporte de progreso
    
    # Detección de recursos de hardware
    n_cores = os.cpu_count()
    
    # --- BUCLE PRINCIPAL DE PROCESAMIENTO ---
    while procesado_actual < TOTAL_MUESTRAS:
        # 1. Calcular el tamaño del lote actual (limitado por LOTE_SEGURO o el remanente)
        restante = TOTAL_MUESTRAS - procesado_actual
        paso_actual = min(restante, LOTE_SEGURO)
        
        # 2. Distribución de carga entre núcleos (Load Balancing)
        chunk_per_core = paso_actual // n_cores
        chunks = [chunk_per_core] * n_cores
        
        # Ajuste de residuos si la división no es exacta
        if sum(chunks) < paso_actual:
            chunks[-1] += (paso_actual - sum(chunks))
            
        # 3. Ejecución Paralela (Modelo Map-Reduce implícito)
        resultados = Parallel(n_jobs=-1)(
            delayed(simular_chunk)(c, n) for c in chunks
        )
        
        # 4. Consolidación de resultados parciales
        for res in resultados:
            total_counts.update(res)
            
        procesado_actual += paso_actual
        
        # --- REPORTE DE ESTADO ---
        porcentaje_actual = (procesado_actual / TOTAL_MUESTRAS) * 100
        if porcentaje_actual >= proximo_hito:
            print(f"   -> Progreso: {int(porcentaje_actual)}% completado...")
            proximo_hito += 10

    # --- EXPORTACIÓN DE RESULTADOS ---
    print("Consolidando datos finales y exportando a Excel...")
    nombre_col_det = f'Det (N={n})'
    nombre_col_freq = f'Freq (1B)'
    
    # Creación del DataFrame de resultados
    df = pd.DataFrame(list(total_counts.items()), columns=[nombre_col_det, nombre_col_freq])
    
    # Cálculo de probabilidad empírica
    df['Probabilidad (%)'] = (df[nombre_col_freq] / TOTAL_MUESTRAS) * 100
    df = df.sort_values(by=nombre_col_det)
    
    # Escritura en hoja de cálculo
    df.to_excel(writer, index=False, startcol=col_inicio)
    print(f"Simulación para N={n} finalizada exitosamente.")
    
    return col_inicio + 4

# --- BLOQUE DE EJECUCIÓN PRINCIPAL ---
if __name__ == '__main__':
    # Definición del rango de estudio: N=1, 2, 3 (Fuerza Bruta implícita) y N=4 (Laplace/Monte Carlo)
    vals = 4 
    N_A_SIMULAR = np.arange(1, vals + 1, 1)
    
    print(f"Iniciando secuencia de simulación para dimensiones: {N_A_SIMULAR}")
    archivo = 'determinantes_1B_safe.xlsx'
    col = 0
    
    tic = time.perf_counter()
    
    # Inicialización del escritor de Excel (motor openpyxl)
    with pd.ExcelWriter(archivo, engine='openpyxl') as writer:
        for n in N_A_SIMULAR:
            # Ejecución iterativa por dimensión
            col = ejecutar_monte_carlo_gigante(n, writer, col)
            
    toc = time.perf_counter()
    
    # Reporte final de métricas de tiempo
    print(f"\n=== TIEMPO TOTAL DE EJECUCIÓN: {toc-tic:.2f} s ({ (toc-tic)/60:.1f} min) ===")

Iniciando secuencia de simulación para dimensiones: [1 2 3 4]

--- INICIANDO SIMULACIÓN PARA N=1 :: OBJETIVO: 1,000,000,000 MUESTRAS ---
   -> Progreso: 10% completado...
   -> Progreso: 20% completado...
   -> Progreso: 30% completado...
   -> Progreso: 40% completado...
   -> Progreso: 50% completado...
   -> Progreso: 60% completado...
   -> Progreso: 70% completado...
   -> Progreso: 80% completado...
   -> Progreso: 90% completado...
   -> Progreso: 100% completado...
Consolidando datos finales y exportando a Excel...
Simulación para N=1 finalizada exitosamente.

--- INICIANDO SIMULACIÓN PARA N=2 :: OBJETIVO: 1,000,000,000 MUESTRAS ---
   -> Progreso: 10% completado...
   -> Progreso: 20% completado...
   -> Progreso: 30% completado...
   -> Progreso: 40% completado...
   -> Progreso: 50% completado...
   -> Progreso: 60% completado...
   -> Progreso: 70% completado...
   -> Progreso: 80% completado...
   -> Progreso: 90% completado...
   -> Progreso: 100% completado...
Consolidan

Cabe notar como se estableció arbitrariamente 1 billón de simulaciones, lo cuál para las primeras tres dimensiones es innecesario, puesto que poseen menos matrices que la muestra tomada. Sin embargo, se vuelve más relevante a partir de la cuarta. El excel genera los datos crudos, pero graficando se puede observar una constancia en la probabilidad de encontrar cualquier valor de determinante, con un aumento de probabilidad conforme se acerca al cero. Para las primeras dimensiones, la determinante cero es una porción sustancial de los posibles valores, pero conforme aumentan las dimensiones se percibe un decrecimiento rápido de la probabilidad de una singularidad. Esto puede relacionarse con estudios previos realizados (al ser este un trabajo preliminar con uso como bitácora, las fuentes no están disponibles dentro del documento, eso sería hecho dentro de un LaTex a futuro):

Los teoremas de Tao y Vu sobre la singularidad de matrices aleatorias discretas se centran principalmente en alfabetos fijos y pequeños (e.g., matrices de Bernoulli con entradas $\pm 1$), demostrando que la probabilidad de singularidad decae exponencialmente ($P_{sing} \le c^N$).Aunque las matrices estudiadas en este trabajo poseen un alfabeto creciente $\mathcal{A}_N = \{0, \dots, N\}$, los teoremas de Tao y Vu son aplicables como una cota superior conservadora. El principio de universalidad sugiere que si la singularidad es asintóticamente rara para alfabetos restringidos, lo es aún más para alfabetos cuyo cardinal crece con $N$, donde la entropía de las filas es mayor y la probabilidad de colisión lineal disminuye drásticamente.



## Categorización Estructural Estricta (N $\leq$ 3)

Para dimensiones pequeñas ($N=1, 2, 3$), podemos contrastar la simulación anterior con la "Verdad Fundamental" obtenida por fuerza bruta exhaustiva.

Con tal de simplificar el análisis a continuación, se establecen arbitrariamente las siguientes abreviaciones para calificar los tipos de matrices que se pueden encontrar:
1. **SM (Same)**: Matrices triviales formadas por un solo escalar repetido.
2. **ZR (Zero Row)**: Se tiene una fila de ceros
3. **ZC (Zero Column)**: Se tiene una columna de zeros
4. **ZRC (Zero Row and Column)**: se tiene una fila y una columna de zeros
5. **ID (Idénticas)**: hay alguna fila repetida
6. **LD (Linealmente Dependiente)**: el residuo, aquellas matrices densas que a su vez son singulares por dependencia aritmética compleja
7. **NON (No singulares)**: Matrices cuya determinante es diferente de cero. Forman parte de las matrices densas

Es de suma importancia mencionar que SM,ZR,ZC,ZRC,ID poseen singularidades estructurales, mientras que no se puede realizar un argumento visual, geométrico o similar de los tipos LD. Las matrices LD y NON conforman las matrices densas. Como se verá en una sección posterior, esto permite generalizar mediante combinatoria el número de matrices singulares por estructura que se obtendrán por dimensión, y se podrá deducir cual será la suma de LD y NON, pero actualmente no es posible determinar con exactitud cuantas de cada una se obtendrán sin realizar el cálculo exhaustivo previamente. Sin embargo, para las primeras tres dimensiones, este cálculo exhaustivo es rápido de realizar, por lo cual se computa directamente.


In [3]:
import numpy as np
import itertools
import time

# =============================================================================
# REPORTE VISUAL
# =============================================================================
def print_report(N, total, sm, zrc, zr, zc, id_count, ld, non, elapsed):
    print(f"\n{'='*60}")
    print(f" PROCESANDO N={N} (FUERZA BRUTA PURA - Sin Fórmulas)")
    print(f"{'='*60}")
    print(f" Tiempo: {elapsed:.4f}s | Espacio Total: {total:,}")
    print(f"{'-'*60}")
    print(f" SM  (Scalar Singular):     {sm:>15,}")
    print(f" ZRC (Row & Col Zero):      {zrc:>15,}")
    print(f" ZR  (Row Zero Only):       {zr:>15,}")
    print(f" ZC  (Col Zero Only):       {zc:>15,}")
    print(f" ID  (Identical Rows):      {id_count:>15,}")
    print(f" LD  (Linear Dependence):   {ld:>15,}")
    print(f" NON (Non-Singular):        {non:>15,}")
    print(f"{'-'*60}")
    
    suma_total = sm + zrc + zr + zc + id_count + ld + non
    check = "OK" if suma_total == total else f"ERROR (Diff: {total - suma_total})"
    print(f" BALANCE TOTAL:             {suma_total:>15,}  [{check}]")

# =============================================================================
# LÓGICA DE CLASIFICACIÓN ITERATIVA
# =============================================================================
def solve_bruteforce_pure(N):
    v = N + 1
    total_matrices = v**(N*N)
    
    # Contadores
    c_sm = 0
    c_zrc = 0
    c_zr = 0
    c_zc = 0
    c_id = 0
    c_ld = 0
    c_non = 0
    
    # Pre-computamos la tupla cero para comparar rápido
    zero_row_tuple = tuple([0] * N)
    
    tic = time.perf_counter()
    
    # --- BUCLE PRINCIPAL: ITERAR SOBRE CADA MATRIZ POSIBLE ---
    # itertools.product genera todas las combinaciones de 0..v-1
    for flat_m in itertools.product(range(v), repeat=N*N):
        
        # 1. Construir matriz y calcular determinante
        arr = np.array(flat_m, dtype=np.int8).reshape(N, N)
        det = np.linalg.det(arr)
        
        # 2. Verificar Singularidad
        # Usamos tolerancia para punto flotante
        if abs(det) > 1e-9:
            c_non += 1
            continue # Si no es singular, pasamos a la siguiente
            
        # --- Si llegamos aquí, ES SINGULAR (Det == 0). Clasificamos: ---
        
        # A. Chequeo SM (Scalar Matrix)
        # Todos los elementos son iguales al primero
        if all(x == flat_m[0] for x in flat_m):
            c_sm += 1
            continue
            
        # B. Chequeo de Ceros
        # Convertimos a tuplas para búsquedas rápidas
        rows = [tuple(r) for r in arr]
        cols = [tuple(c) for c in arr.T]
        
        has_zr = zero_row_tuple in rows
        has_zc = zero_row_tuple in cols
        
        if has_zr and has_zc:
            c_zrc += 1
            continue
        elif has_zr:
            c_zr += 1
            continue
        elif has_zc:
            c_zc += 1
            continue
            
        # C. Chequeo ID (Identical Rows)
        # Solo revisamos filas, tal como en tu lógica original.
        # Si el número de filas únicas es menor a N, hay repetidas.
        if len(set(rows)) < N:
            c_id += 1
            continue
            
        # D. LD (Linear Dependence)
        # Si es singular y no cayó en ninguna anterior, es LD pura.
        c_ld += 1

    elapsed = time.perf_counter() - tic
    print_report(N, total_matrices, c_sm, c_zrc, c_zr, c_zc, c_id, c_ld, c_non, elapsed)

if __name__ == "__main__":
    # Ejecutamos para N=1, 2, 3
    # NOTA: N=3 toma unos segundos porque analiza 262,144 matrices una por una.
    for n in range(1, 4):
        solve_bruteforce_pure(n)


 PROCESANDO N=1 (FUERZA BRUTA PURA - Sin Fórmulas)
 Tiempo: 0.0001s | Espacio Total: 2
------------------------------------------------------------
 SM  (Scalar Singular):                   1
 ZRC (Row & Col Zero):                    0
 ZR  (Row Zero Only):                     0
 ZC  (Col Zero Only):                     0
 ID  (Identical Rows):                    0
 LD  (Linear Dependence):                 0
 NON (Non-Singular):                      1
------------------------------------------------------------
 BALANCE TOTAL:                           2  [OK]

 PROCESANDO N=2 (FUERZA BRUTA PURA - Sin Fórmulas)
 Tiempo: 0.0010s | Espacio Total: 81
------------------------------------------------------------
 SM  (Scalar Singular):                   3
 ZRC (Row & Col Zero):                    8
 ZR  (Row Zero Only):                     8
 ZC  (Col Zero Only):                     8
 ID  (Identical Rows):                    2
 LD  (Linear Dependence):                 2
 NON (Non-Singular

## Categorización Estructural Estricta (N = 4)

Para $N=4$, el espacio es de $1.52 \times 10^{11}$ matrices. La fuerza bruta pura toma en este computador alrededor de 33 horas, o aun menos con un computador con más núcleos. Sin embargo, esto es vastamente ineficiente, entonces se realiza lo siguiente:

**Expansión de Cofactores**
En lugar de generar matrices completas $4 \times 4$, generamos geometrías base de 3 filas ($N \times (N-1)$).

* Pre-cálculo de Cofactores: Para cada conjunto de 3 filas, calculamos los 4 cofactores que resultan de expandir el determinante a lo largo de la (hipotética) cuarta fila. Esto reduce el cálculo del determinante a un producto punto:$$\det(A) = \vec{r}_4 \cdot \vec{C}_{base}$$

* Vectorización Masiva: Multiplicamos el vector de cofactores $\vec{C}_{base}$ contra todas las posibles filas cuartas ($\vec{r}_4 \in P^4$) en una sola operación matricial.

* Generadores Lazy: Usamos itertools.combinations_with_replacement sobre las filas únicas. Esto comprime el espacio de búsqueda significativamente porque no nos importa el orden de las filas para calcular la singularidad (solo sus valores), y luego ajustamos los conteos usando pesos multinomiales.

In [7]:
import numpy as np
import itertools
import math
import time
from collections import Counter
from joblib import Parallel, delayed

# --- CONFIGURACIÓN ---
N = 4
V = 5  # Valores 0..4

def get_unique_permutations_count(indices):
    """
    Versión optimizada de conteo de permutaciones para tupla de 4 elementos.
    Evita overhead de Counter.
    """
    # Ordenamos para comparar
    a, b, c, d = indices # Ya vienen ordenados r1<=r2<=r3<=r4 casi siempre
    
    # Lógica hardcodeada para N=4 es más rápida que loops genéricos
    if a == d: return 1           # (x,x,x,x) -> 1
    if a == c or b == d: return 4 # (x,x,x,y) o (x,y,y,y) -> 4
    if a == b and c == d: return 6 # (x,x,y,y) -> 6
    if a == b or b == c or c == d: return 12 # (x,x,y,z), (x,y,y,z), (x,y,z,z) -> 12
    return 24 # (x,y,z,w) -> 24

def procesar_chunk_turbo(chunk_indices, all_rows, row_masks, row_is_uniform, idx_zero_row):
    """
    Chunk optimizado con operaciones de bits y sin creación de arrays numpy.
    """
    local_counts = np.zeros(7, dtype=np.int64) 
    # Indices: 0:SM, 1:ZRC, 2:ZR, 3:ZC, 4:ID, 5:LD, 6:NON (no usado aquí)
    
    for idxs_3 in chunk_indices:
        r1, r2, r3 = idxs_3
        
        # 1. Construir matriz base 3x4 (Solo aquí usamos numpy)
        # Esto es inevitable para el determinante, pero es vectorizado
        block_3 = all_rows[[r1, r2, r3]]
        
        # 2. Calcular Cofactores (Laplace)
        # Optimizamos extrayendo columnas directamente
        c0 = block_3[:, 0]
        c1 = block_3[:, 1]
        c2 = block_3[:, 2]
        c3 = block_3[:, 3]
        
        # Determinantes 3x3 hardcodeados (Más rápido que np.linalg.det repetido)
        # Cofactor 0 (cols 1,2,3)
        m0 = c1[0]*(c2[1]*c3[2] - c2[2]*c3[1]) - c1[1]*(c2[0]*c3[2] - c2[2]*c3[0]) + c1[2]*(c2[0]*c3[1] - c2[1]*c3[0])
        # Cofactor 1 (cols 0,2,3)
        m1 = c0[0]*(c2[1]*c3[2] - c2[2]*c3[1]) - c0[1]*(c2[0]*c3[2] - c2[2]*c3[0]) + c0[2]*(c2[0]*c3[1] - c2[1]*c3[0])
        # Cofactor 2 (cols 0,1,3)
        m2 = c0[0]*(c1[1]*c3[2] - c1[2]*c3[1]) - c0[1]*(c1[0]*c3[2] - c1[2]*c3[0]) + c0[2]*(c1[0]*c3[1] - c1[1]*c3[0])
        # Cofactor 3 (cols 0,1,2)
        m3 = c0[0]*(c1[1]*c2[2] - c1[2]*c2[1]) - c0[1]*(c1[0]*c2[2] - c1[2]*c2[0]) + c0[2]*(c1[0]*c2[1] - c1[1]*c2[0])

        # Vector Normal C = [m0, -m1, m2, -m3]
        # Producto punto masivo: all_rows @ C
        # Expandimos: r[:,0]*m0 - r[:,1]*m1 + r[:,2]*m2 - r[:,3]*m3
        dots = all_rows[:,0]*m0 - all_rows[:,1]*m1 + all_rows[:,2]*m2 - all_rows[:,3]*m3
        
        # Filtrar candidatos (Producto punto == 0)
        valid_r4 = np.where(dots == 0)[0]
        
        # Filtrar por orden canónico (r4 >= r3)
        # Esto reduce drásticamente el bucle siguiente
        valid_r4 = valid_r4[valid_r4 >= r3]
        
        # --- BUCLE INTERNO ULTRA RÁPIDO ---
        # Pre-cargamos propiedades de las filas fijas
        mask_123 = row_masks[r1] & row_masks[r2] & row_masks[r3]
        has_zr_123 = (r1 == idx_zero_row) or (r2 == idx_zero_row) or (r3 == idx_zero_row)
        
        for r4 in valid_r4:
            # Peso combinatorio
            weight = get_unique_permutations_count((r1, r2, r3, r4))
            
            # --- CLASIFICACIÓN JERÁRQUICA (Sin NumPy) ---
            
            # 1. SM (Scalar Matrix)
            # Solo si r1=r2=r3=r4 y además es fila uniforme
            if r1 == r4: # Implica r1=r2=r3=r4 porque r1<=r2<=r3<=r4
                if row_is_uniform[r1]:
                    local_counts[0] += weight # SM
                    continue

            # 2. ZRC / ZR / ZC
            # Bitwise check para columnas cero
            has_zc = (mask_123 & row_masks[r4]) > 0
            has_zr = has_zr_123 or (r4 == idx_zero_row)
            
            if has_zr and has_zc:
                local_counts[1] += weight # ZRC
                continue
            if has_zr:
                local_counts[2] += weight # ZR
                continue
            if has_zc:
                local_counts[3] += weight # ZC
                continue
                
            # 3. ID (Identical Rows)
            # Si hay repetidos en (r1, r2, r3, r4).
            # Como están ordenados, solo chequeamos vecinos
            if r1 == r2 or r2 == r3 or r3 == r4:
                local_counts[4] += weight # ID
                continue
                
            # 4. LD (Linear Dependence)
            local_counts[5] += weight # LD

    return local_counts

def ejecutar_turbo_n4():
    print(f"\n{'='*60}")
    print(f" MODO TURBO N=4 (Validación Numérica)")
    print(f"{'='*60}")
    
    tic = time.perf_counter()
    
    # 1. Generar Universo y Pre-cálculos
    vals = np.arange(V, dtype=np.int8)
    all_rows = np.array(list(itertools.product(vals, repeat=N)), dtype=np.int64) # int64 para evitar overflow en dots
    n_rows = len(all_rows)
    
    # A. Máscaras de Ceros (Bitwise)
    # [0, 2, 0, 1] -> 1010 -> Si hay un cero, ponemos 1 en la mascara para AND
    # Espera, para ZC necesitamos que la columna SEA cero.
    # Máscara: 1 si es cero, 0 si tiene valor.
    # Row: [0, 2, 0, 1] -> Mask: [1, 0, 1, 0] (binario 10)
    row_masks = np.zeros(n_rows, dtype=np.int32)
    for i in range(n_rows):
        mask = 0
        for j in range(N):
            if all_rows[i, j] == 0:
                mask |= (1 << j)
        row_masks[i] = mask
        
    # B. Uniformidad
    row_is_uniform = np.array([len(set(row)) == 1 for row in all_rows], dtype=bool)
    
    # C. Índice Fila Cero
    idx_zero_row = -1
    for i in range(n_rows):
        if np.all(all_rows[i] == 0):
            idx_zero_row = i
            break
            
    print(f" Universo preparado. Filas: {n_rows}")
    
    # 2. Generador de Combinaciones
    # Reducimos un poco el Batch Size para ver actualizaciones más seguido
    comb_iter = itertools.combinations_with_replacement(range(n_rows), 3)
    BATCH_SIZE = 50_000 
    
    def chunk_generator():
        while True:
            chunk = tuple(itertools.islice(comb_iter, BATCH_SIZE))
            if not chunk: break
            yield chunk
            
    # 3. Ejecución Paralela
    print(" Iniciando Turbo-Scan...")
    
    # verbose=10 imprime mucho, usaremos verbose=5
    results = Parallel(n_jobs=-1, verbose=5)(
        delayed(procesar_chunk_turbo)(chunk, all_rows, row_masks, row_is_uniform, idx_zero_row) 
        for chunk in chunk_generator()
    )
    
    # 4. Consolidación
    final_arr = np.sum(results, axis=0)
    cats = ['SM', 'ZRC', 'ZR', 'ZC', 'ID', 'LD']
    final_counts = dict(zip(cats, final_arr))
    
    elapsed = time.perf_counter() - tic
    
    # 5. Reporte
    total_space = V**(N*N)
    total_singular = sum(final_counts.values())
    final_counts['NON'] = total_space - total_singular
    
    print(f"\n{'='*60}")
    print(f" RESULTADOS FINALES TURBO N=4")
    print(f" Tiempo: {elapsed/60:.2f} minutos")
    print(f"{'-'*60}")
    
    orden = ['SM', 'ZRC', 'ZR', 'ZC', 'ID', 'LD', 'NON']
    suma_check = 0
    
    for cat in orden:
        val = final_counts[cat]
        suma_check += val
        pct = (val / total_space) * 100
        print(f" {cat:<5}: {val:>18,}  ({pct:.6f}%)")
        
    print(f"{'-'*60}")
    print(f" TOTAL : {suma_check:>18,}")
    print(f" ESPACIO: {total_space:>18,}")
    
    if total_space == suma_check:
        print(" [OK] BALANCE PERFECTO")
    else:
        print(f" [!] DIFERENCIA: {total_space - suma_check}")

if __name__ == '__main__':
    ejecutar_turbo_n4()


 MODO TURBO N=4 (Validación Numérica)
 Universo preparado. Filas: 625
 Iniciando Turbo-Scan...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   21.6s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 154 tasks      | elapsed:  2.4min
[Parallel(n_jobs=-1)]: Done 280 tasks      | elapsed:  3.9min
[Parallel(n_jobs=-1)]: Done 442 tasks      | elapsed:  5.8min
[Parallel(n_jobs=-1)]: Done 640 tasks      | elapsed:  7.9min



 RESULTADOS FINALES TURBO N=4
 Tiempo: 9.81 minutos
------------------------------------------------------------
 SM   :                  5  (0.000000%)
 ZRC  :         30,525,376  (0.020005%)
 ZR   :        943,695,872  (0.618461%)
 ZC   :        943,695,872  (0.618461%)
 ID   :      1,408,918,524  (0.923349%)
 LD   :      5,963,540,208  (3.908266%)
 NON  :    143,297,514,768  (93.911459%)
------------------------------------------------------------
 TOTAL :    152,587,890,625
 ESPACIO:    152,587,890,625
 [OK] BALANCE PERFECTO


[Parallel(n_jobs=-1)]: Done 818 out of 818 | elapsed:  9.8min finished


## Generalización por combinatoria para singularidades estructurales

Concisamente, se utiliza teoría de conjuntos y el principio de Inclusión-Exclusión para obtener fórmulas que generalicen la cantidad de singularidades estructurales a gran detalle, dejando LD y NON dentro de un grupo que actualmente no se puede separar. Para esta sección, se define formalmente $v=N+1$, tal que la cardinalidad para una dimensión $N$ es $|\Omega|=T=v^{N^2}$

### 1. **Matrices Densas ($K_{dense}$)**: 
Es la cardinalidad del conjunto de matrices densas estructurales ($\mathcal{D}$) que no tienen ninguna fila nula ni ninguna columna nula. Para calcularlo, fijamos primero el universo de matrices que no tienen filas nulas: $(v^N - 1)^N$ (ya que cada una de las $N$ filas debe ser un vector no nulo). Sobre este universo, aplicamos PIE para excluir las matrices con columnas nulas.

$$\mathcal{D} = A^c \cap B^c$$

Siendo $B$ el evento describiendo cuando se tiene al menos una columna nula. Sea $P_j$ la propiedad de que la columna $j$ es nula. Buscamos el número de elementos en $A^c$ que no satisfacen ninguna $P_j$.

$$K = \sum_{k=0}^N (-1)^k S_k$$

Donde $S_k$ es la suma de los tamaños de las intersecciones de $k$ columnas fijadas a cero. Si se fijan $k$ columnas específicas a cero:

1. **Restricción de Columnas**: Estas $k$ columnas son obligatoriamente $\vec{0}$

2. **Grados de Libertad**: Nos quedan $N-k$ columnas libres

3. **Restricción de Filas ($A^c$)**: La matriz resultante no puede tener filas nulas
Al fijar $k$ columnas a cero, cada fila de la matriz se convierte efectivamente en un vector de dimensión $N-k$. Para que la fila original no sea nula, este sub-vector de dimensión $N-k$ no puede ser todo ceros. Entonces, el número de vectores válidos de longitud $N-k$ es: $v^{N-k} - 1$.
4. **Independencia**: Como elegimos las $N$ filas independientemente:
$$\text{Espacio válido dado } k \text{ columnas nulas} = (v^{N-k} - 1)^N$$
5. Multiplicando por las formas de elegir las columnas $\binom{N}{k}$ y el signo alternante, se obtiene lo siguiente:
   $$K = \sum_{k=0}^N (-1)^k \binom{N}{k} (v^{N-k} - 1)^N$$

### 2. Matrices con singularidades estructurales involucrando el cero

Se deduce que la unión de singularidades estructurales es el complemento de las densas:
$$|\mathcal{R} \cup \mathcal{C}| = T - K$$

Queremos la intersección $I_{rc} = |\mathcal{R} \cap \mathcal{C}|$ (matrices con filas Y columnas nulas). Por la identidad fundamental de conjuntos $|A \cup B| = |A| + |B| - |A \cap B|$, despejamos la intersección:
$$|\mathcal{R} \cap \mathcal{C}| = |\mathcal{R}| + |\mathcal{C}| - |\mathcal{R} \cup \mathcal{C}|$$

Sustituyendo (y usando simetría $|\mathcal{R}| = |\mathcal{C}| = U_r$):

$$I_{rc} = 2 U_r - (T - K)$$

Con todos estos componentes, se tiene suficiente información para obtener los tipos ZR,ZC y ZRC.

1. **SM**: Es el caso más trivial
$$\text{SM}=v$$
3. **ZR**: Matrices con filas nulas que no tienen columnas nulas
   $$ZR = U_r - I_{rc}$$
5. **ZC**: Matrices con columnas nulas que no tienen filas nulas
   $$ZC = U_c - I_{rc}$$
7. **ZRC**:Es la intersección pura, pero restando 1 (Con tal de no contar las matrices SM dos veces)
   $$ZRC = I_{rc} - 1$$


Ahora, es importante considerar lo siguiente para los cálculos en sí:
* Debido a la simetría de las matrices, $U_r=U_c$
* $U_r =U_c= T - (v^N - 1)^N$
* $I_{rc} = 2 U_r - T + K$



### 3. Singularidades de matrices con filas identicas (ID)
La fórmula para ID no es trivial porque no podemos simplemente contar matrices con filas repetidas; debemos contar solo aquellas que son densas (sin ceros) y que no son escalares (SM). Para lograr esto, mapeamos el problema a particiones de conjuntos.

1. **Particiones (Números de Bell)**: Las filas repetidas definen una relación de equivalencia entre los índices de las filas. Si la fila 1 es igual a la fila 2, los índices $\{1, 2\}$ están en el mismo bloque. Iteramos sobre todas las particiones $\pi$ del conjunto $\{1, \dots, N\}$.
2. **Coeficiente de Möbius:** Usamos un coeficiente $\mu(\pi)$ derivado del retículo de particiones para corregir el sobreconteo inherente (ej. si fila 1=2=3, esto se cuenta en 1=2, 2=3 y 1=3).
   $$\mu(\pi) = (-1)^{N-|\pi|-1} \prod_{B \in \pi} (|B|-1)!$$
3. **Inclusión-Exclusión Interna**: Para una partición dada con $k$ bloques (es decir, $k$ filas únicas efectivas), calculamos cuántas matrices densas existen. Tratamos la matriz comprimida como una matriz de $k \times N$. Aplicamos un PIE (Principio Inclusión-Exclusión) interno sobre esta matriz para asegurar que ninguna columna sea nula (usando el alfabeto de columnas $v^{k-s}-1$, donde $s$ son las filas forzadas a cero)
$$ID = \left( \sum_{\pi \in \Pi_N, |\pi|<N} \mu(\pi) \cdot \text{Dense}(\pi) \right) - (v-1)$$

Restamos $(v-1)$ al final para remover las matrices constantes no nulas (SM), que matemáticamente cumplen con ser densas e idénticas, pero queremos categorizarlas aparte.

Con esto, se puede generalizar el número de matrices que se observan en cada dimensión por tipo, a excepción de LD y NON, que tienen que ser agrupadas dentro del conjunto de matrices densas. El código siguiente muestra los cálculos generalizados, y los resultados corroboran lo obtenido en secciones previas.

In [2]:
import math

# =============================================================================
# 1. HERRAMIENTAS COMBINATORIAS
# =============================================================================

def generate_partitions(collection):
    """Genera particiones de un conjunto (Bell numbers)."""
    if len(collection) == 1:
        yield [collection]
        return
    first = collection[0]
    for smaller in generate_partitions(collection[1:]):
        for n, subset in enumerate(smaller):
            yield smaller[:n] + [[first] + subset]  + smaller[n+1:]
        yield [[first]] + smaller

def calculate_partition_coefficient(partition, N):
    """Coeficiente de Inclusión-Exclusión (Möbius) para el retículo."""
    k = len(partition)
    sign = (-1)**(N - k - 1)
    block_factor = 1
    for block in partition:
        block_factor *= math.factorial(len(block) - 1)
    return sign * block_factor

# =============================================================================
# 2. FÓRMULAS GENERALES EXACTAS
# =============================================================================

def calculate_exact_distribution(max_n):
    print(f"{'='*80}")
    print(f" ESTIMACIÓN MATRICIAL GENERAL (Alineada con Fuerza Bruta N=1..3)")
    print(f"{'='*80}")

    for N in range(1, max_n + 1):
        v = N + 1
        T_total = v**(N*N)
        
        # --- 1. SM (Scalar Matrix) ---
        SM = v if N > 1 else 1

        # --- 2. CEROS (ZRC, ZR, ZC) ---

        K_dense = 0
        for k in range(N + 1):
            term = math.comb(N, k) * ((-1)**k) * ((v**(N-k) - 1)**N)
            K_dense += term
            
        # Conjuntos de Ceros
        Rows_Zero = T_total - ((v**N - 1)**N) # Total - (Filas No Cero)^N
        Cols_Zero = Rows_Zero # Simetría
        
        # Intersección y Unión
        # Union(ZR, ZC) = Total - K_dense
        # Intersection(ZR, ZC) = |ZR| + |ZC| - Union
        
        Union_Z = T_total - K_dense
        Inter_Z = 2 * Rows_Zero - Union_Z
        
        ZRC = Inter_Z - 1      # Restamos la matriz nula (contada en SM)
        ZR = Rows_Zero - Inter_Z
        ZC = Cols_Zero - Inter_Z

        # --- 3. ID (Identical Rows - Strict Dense) ---
        # Calculamos matrices con filas repetidas dentro del Universo Denso.
        ID_raw = 0
        if N > 1:
            rows_indices = list(range(N))
            for part in generate_partitions(rows_indices):
                k = len(part)
                if k == N: continue # Ignorar filas distintas

                coeff = calculate_partition_coefficient(part, N)
                
                # Inclusión-Exclusión Interna (Dense Universe Logic)
                # Para una matriz comprimida k*N, asegurar No Row Zero y No Col Zero.
                count_dense_struct = 0
                for s in range(k + 1): # s = filas comprimidas forzadas a cero
                    sign = (-1)**s
                    ways_to_zero = math.comb(k, s)
                    
                    # Alfabeto columna efectivo: v^(k-s) - 1 (para no tener col ceros)
                    # Pero debemos elevar a la N (N columnas)
                    eff_alphabet = (v**(k - s)) - 1
                    
                    if eff_alphabet <= 0:
                        term = 0
                    else:
                        term = eff_alphabet**N
                    
                    count_dense_struct += sign * ways_to_zero * term
                
                ID_raw += coeff * count_dense_struct
        
        # (Solo si N > 1, si N=1 ID es 0).
        sm_correction = (v - 1) if N > 1 else 0
        ID = ID_raw - sm_correction
        if ID < 0: ID = 0

        # --- 4. RESTO (LD + NON) ---
        # Todo lo que no es fórmula estructural
        clasificado = SM + ZRC + ZR + ZC + ID
        resto = T_total - clasificado

        # --- REPORTE ---
        print(f"\n>>> DIMENSIÓN N={N} (Total={T_total:,})")
        print(f"{'-'*60}")
        print(f"   SM  (Scalar):             {SM:>15,}")
        print(f"   ZRC (Zero Row & Col):     {ZRC:>15,}")
        print(f"   ZR  (Zero Row Only):      {ZR:>15,}")
        print(f"   ZC  (Zero Col Only):      {ZC:>15,}")
        print(f"   ID  (Identical R/C):      {ID:>15,}")
        print(f"{'-'*60}")
        print(f"   SUMA CLASIFICADA:         {clasificado:>15,}")
        print(f"   RESTO (LD + NON):         {resto:>15,}  <-- RESULTADO FINAL")
        print(f"{'='*60}")
        
        # Verificación automática
        if N == 1: check_vals(N, resto, 1)
        if N == 2: check_vals(N, resto, 2 + 50) # LD+NON = 52
        if N == 3: check_vals(N, resto, 17196 + 212898) # LD+NON = 230094
        if N == 3: check_vals(N, ID, 9906, "ID") # Check específico ID

def check_vals(N, val, expected, label="Resto"):
    res = "CORRECTO" if val == expected else f"FALLO (Esp: {expected})"
    print(f"   [Verificación N={N} {label}]: {res}")

# Ejecutar hasta N=7
calculate_exact_distribution(7)

 ESTIMACIÓN MATRICIAL GENERAL (Alineada con Fuerza Bruta N=1..3)

>>> DIMENSIÓN N=1 (Total=2)
------------------------------------------------------------
   SM  (Scalar):                           1
   ZRC (Zero Row & Col):                   0
   ZR  (Zero Row Only):                    0
   ZC  (Zero Col Only):                    0
   ID  (Identical R/C):                    0
------------------------------------------------------------
   SUMA CLASIFICADA:                       1
   RESTO (LD + NON):                       1  <-- RESULTADO FINAL
   [Verificación N=1 Resto]: CORRECTO

>>> DIMENSIÓN N=2 (Total=81)
------------------------------------------------------------
   SM  (Scalar):                           3
   ZRC (Zero Row & Col):                   8
   ZR  (Zero Row Only):                    8
   ZC  (Zero Col Only):                    8
   ID  (Identical R/C):                    2
------------------------------------------------------------
   SUMA CLASIFICADA:             